# Submissao 2 - DNN + ForensicVectorizer

Treino completo com os melhores hiperparâmetros encontrados e predição sobre `subm2.csv`.

**Modelo:** DNN NumPy com ForensicVectorizer (POS n-grams + function words + stylometric features)

**Melhores params:** hidden=[64, 32, 16], dropout=0.4, lr=0.003, batch_size=64, max_words=1500

In [ ]:
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from src.vectorizer import ForensicVectorizer
from src.models_numpy.dnn.neuralnet import NeuralNetwork
from src.models_numpy.dnn.layers import DenseLayer, DropoutLayer
from src.models_numpy.dnn.activation import ReLUActivation, SoftmaxActivation
from src.models_numpy.dnn.losses import CategoricalCrossEntropy
from src.models_numpy.dnn.metrics import accuracy
from src.models_numpy.dnn.dataset import Dataset
from src.models_numpy.dnn.optimizer import AdamOptimizer

train_csv = Path('../data/processed/dataset_combined.csv')
input_csv = Path('validation_data/subm2.csv')
output_csv = Path('subm2-g8-MEI-A.csv')

CLASSES = ["Anthropic", "Google", "Human", "Meta", "OpenAI"]
LABEL_MAP = {label: i for i, label in enumerate(CLASSES)}
IDX_TO_LABEL = {i: label for label, i in LABEL_MAP.items()}
NUM_CLASSES = len(CLASSES)
SEED = 42

HIDDEN_LAYERS = [64, 32, 16]
DROPOUT = 0.4
LR = 0.003
BATCH_SIZE = 64
MAX_WORDS = 1500

np.random.seed(SEED)

print("Config loaded.")

In [ ]:
df_train = pd.read_csv(train_csv, sep=';')
df_train = df_train[df_train['Label'].isin(CLASSES)].copy()
df_train['label_id'] = df_train['Label'].map(LABEL_MAP)

print(f"Train samples: {len(df_train)}")
print(f"Label distribution:\n{df_train['Label'].value_counts()}")

raw_train = list(df_train['Text'])
y_train = df_train['label_id'].values

# One-hot encode
y_train_oh = np.zeros((len(y_train), NUM_CLASSES))
for i, l in enumerate(y_train):
    y_train_oh[i, l] = 1

In [ ]:
print("Fitting ForensicVectorizer (max_words={})...".format(MAX_WORDS))
vec = ForensicVectorizer(max_words=MAX_WORDS)
X_all = vec.fit_transform([""] * len(raw_train), raw_train)
print(f"Feature matrix: {X_all.shape}")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_train_oh, test_size=0.2, random_state=SEED, stratify=y_train
)
print(f"Train: {X_tr.shape[0]}, Val: {X_te.shape[0]}")

In [ ]:
opt = AdamOptimizer(learning_rate=LR)
model = NeuralNetwork(
    epochs=100, batch_size=BATCH_SIZE, optimizer=opt,
    loss=CategoricalCrossEntropy, metric=accuracy,
    early_stopping=True, patience=15, verbose=True,
)

input_dim = X_tr.shape[1]
prev_dim = input_dim
for h in HIDDEN_LAYERS:
    model.add(DenseLayer(h, prev_dim))
    model.add(ReLUActivation())
    if DROPOUT > 0:
        model.add(DropoutLayer(DROPOUT))
    prev_dim = h
model.add(DenseLayer(NUM_CLASSES, prev_dim))
model.add(SoftmaxActivation())

print(f"Architecture: {input_dim} -> {' -> '.join(str(h) for h in HIDDEN_LAYERS)} -> {NUM_CLASSES}")
print("Training...")
model.fit(Dataset(X_tr, y_tr), Dataset(X_te, y_te))

In [ ]:
preds_val = np.argmax(model.predict(Dataset(X_te, y_te)), axis=1)
y_te_ids = np.argmax(y_te, axis=1)
val_acc = accuracy_score(y_te_ids, preds_val)
print(f"\nInternal validation accuracy: {val_acc:.4f}")
print(classification_report(y_te_ids, preds_val, target_names=CLASSES, zero_division=0))

In [ ]:
df_sub = pd.read_csv(input_csv, sep=';')
print(f"Submission samples: {len(df_sub)}")

raw_sub = list(df_sub['Text'].fillna(''))
X_sub = vec.transform([""] * len(raw_sub), raw_sub)
print(f"Submission features: {X_sub.shape}")

dummy_y = np.zeros((len(raw_sub), NUM_CLASSES))
pred_ids = np.argmax(model.predict(Dataset(X_sub, dummy_y)), axis=1)

df_sub['Label'] = [IDX_TO_LABEL[int(i)] for i in pred_ids]
df_out = df_sub[['ID', 'Label']]
df_out.to_csv(output_csv, sep=';', index=False)

print(f"\nSubmissao guardada em: {output_csv}")
print(f"Distribuicao:\n{df_out['Label'].value_counts()}")
print(df_out.head(10))